# pymefpp_create_associations

This notebook is used to create the associations required to run mefpp


# Testing notebook

It works without any problem or warning

In [1]:
import os
import sys
import glob
import logging

from pathlib import Path

# here we import the local mefpp_run library
sys.path.insert(0, os.getcwd())
from mefpp_associate_reduction import *

In [2]:
# we set the data reduction directories

root_dir = "/Users/fselman/Pictures/OCAM/CPPup/reduced/"
raw_data_dir = "/Users/fselman/Pictures/OCAM/CPPup/raw/"
reduced_cal_dir = root_dir+'calibs/'
reduced_science_dir = root_dir+'science/'
log_dir = root_dir+'logs/'

# we prepare the logs
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_dir+"/pymeffpp_associate.log"),
                              logging.StreamHandler()
                    ])

In [3]:
bpm_list = glob.glob(reduced_cal_dir+'**/bpm.list', recursive=True)
logging.info('number of bpm partitions: {0}'.format(len(bpm_list)))
bpm_partitions = []
for bpm_file in bpm_list:
    bpm_partitions.append(BiasPartition(bpm_file))
    
for part in bpm_partitions:
    logging.info(part.OBstart)

2024-05-23 14:33:24,230:INFO:number of bpm partitions: 1
2024-05-23 14:33:24,231:INFO:bpm


In [4]:
bias_list = glob.glob(reduced_cal_dir+'**/bias.list', recursive=True)
print('number of bias partitions: ', len(bias_list))
bias_partitions = []
for bias_file in bias_list:
    bias_partitions.append(BiasPartition(bias_file))
    
for part in bias_partitions:
    print(part.OBstart)

number of bias partitions:  2
2017-03-13T09:44:19
2017-03-14T09:41:34


In [5]:
dome_list = glob.glob(reduced_cal_dir+'**/dome.list', recursive=True)
print('number of dome partitions: ', len(dome_list))
dome_partitions = []
for dome_file in dome_list:
    dome_partitions.append(SciencePartition(dome_file))
    
for part in dome_partitions:
    print(part.OBstart, part.filter)

number of dome partitions:  1
2017-03-14T09:41:34 r_SDSS


In [6]:
sky_list = glob.glob(reduced_cal_dir+'**/sky.list', recursive=True)
print('number of sky partitions: ', len(sky_list))
sky_partitions = []
for sky_file in sky_list:
    sky_partitions.append(SciencePartition(sky_file))
    
for part in sky_partitions:
    print(part.OBstart, part.filter)

number of sky partitions:  1
2017-03-13T23:12:13 r_SDSS


In [7]:
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))

sci_partitions = []
for science_file in science_list:
    sci_partitions.append(SciencePartition(science_file))
    
for part in sci_partitions:
    print(part.OBstart, part.filter)



number of science partitions:  1
2017-03-13T01:57:49 r_SDSS


In [8]:
#We associate a bpm partition to each science frame
for sci_part in sci_partitions:
    sci_part.bpmPartition = bpm_partitions[0]
    for i in range(1,33):
        bpm_file = (sci_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sci_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sci_part.bpmPartition).base_path+'/toreduce_sci').touch()
        print(bpm_file, sci_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))


/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_01.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/science/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGACAM_BPM_01.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_02.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/science/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGACAM_BPM_02.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_03.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/science/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGACAM_BPM_03.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_04.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/science/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OMEGACAM_BPM_04.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_05.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/science/science/CP_Pup_081146p060_m352105p

In [9]:
# we associate a bias partition to each science partition

for sci_part in sci_partitions:
    # here we look for the closest in time bias partition
    sci_date = sci_part.OBstart

    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(sci_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sci_part.biasPartition = bias_part
    
    # we find the sky closest in time and we link the mff to a file that
    # will be present there once it is created
    #
    skyBestDeltaT = 3600.*24.*30.0
    for sky_part in sky_partitions:
        if sci_part.filter != sky_part.filter:
            continue
        sky_date = sky_part.OBstart
        deltaT = timeDistance(sci_date, sky_date)

        if deltaT < skyBestDeltaT:
            skyBestDeltaT = deltaT
            sci_part.skyPartition = sky_part
            sci_part.mffPartition = sky_part

    
    print('science: %s %s'%(sci_part.filter, sci_part.OBstart,)+\
            '     bias: %s (%3.1f)'%(sci_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' sky: %s %s (%3.1f)'%(sci_part.skyPartition.filter, sci_part.skyPartition.OBstart, skyBestDeltaT/3600.0/24.,))

# at this point we have the biases and mff for each science frame. Now we link
for sci_part in sci_partitions:
    mbias = (sci_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sci_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((sci_part.biasPartition).base_path+'/toreduce_sci').touch()
        print(mbias)

    mff = (sci_part.mffPartition).base_path+'/OMEGACAM_MFF_'+sci_part.filter+'_1x1.fits'
    if not Path(mff).is_file():
        try:
            os.symlink(mff, sci_part.base_path+'/OMEGACAM_MFF_'+sci_part.filter+'_1x1.fits')
        except:
            pass
        Path((sci_part.mffPartition).base_path+'/toreduce_mff').touch()


science: r_SDSS 2017-03-13T01:57:49     bias: 2017-03-13T09:44:19 (0.3) sky: r_SDSS 2017-03-13T23:12:13 (0.9)
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bias/574184/2017-03-13T09:44:19/OMEGACAM_MB_1x1.fits


In [10]:
#We associate a bpm partition to each sky frame partition
for sky_part in sky_partitions:
    sky_part.bpmPartition = bpm_partitions[0]
    for i in range(1,33):
        bpm_file = (sky_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sky_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sky_part.bpmPartition).base_path+'/toreduce_sky').touch()
        print(bpm_file, sky_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))

/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_01.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_01.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_02.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_02.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_03.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_03.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_04.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_04.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BPM_05.fits /Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/sky/200253584/r_SDSS/2017-03-13T23:12:13/OMEGACAM_BPM_05.fits
/Users/fselman/Pictures/OCAM/CPPup/reduced/calibs/bpm/OMEGACAM_BP

In [11]:
# here we associate dome and bias to each sky partition
for sky_part in sky_partitions:
    sky_date = sky_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(sky_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sky_part.biasPartition = bias_part
    
    # we find the dome closest in time and we link the domeff to a file that
    # will be present there once it is created
    #
    domeBestDeltaT = 3600.*24.*30.0
    for dome_part in dome_partitions:
        if sky_part.filter == dome_part.filter:
            dome_date = dome_part.OBstart
            deltaT = timeDistance(sky_date, dome_date)
            #print(sci_date, bias_date, deltaT)
            if deltaT < domeBestDeltaT:
                domeBestDeltaT = deltaT
                sky_part.domePartition = dome_part
        else:
            pass

    
    print('sky: %s %s'%(sky_part.filter, sky_part.OBstart,)+\
            '     bias: %s (%4.2f)'%(sky_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' dome: %s %s (%4.2f)'%(sky_part.domePartition.filter, sky_part.domePartition.OBstart, domeBestDeltaT/3600.0/24.,))
    
# at this point we have the biases and domes for each sky ff frame. Now we link
for sky_part in sky_partitions:
    mbias = (sky_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sky_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((sky_part.biasPartition).base_path+'/toreduce_sky').touch()
        
    domeff = (sky_part.domePartition).base_path+'/OMEGACAM_DOMEFF_'+sky_part.filter+'_1x1.fits'
    if not Path(domeff).is_file():
        try:
            os.symlink(domeff, sky_part.base_path+'/OMEGACAM_DOMEFF_'+sky_part.filter+'_1x1.fits')
        except:
            pass
        Path((sky_part.domePartition).base_path+'/toreduce_sky').touch()


sky: r_SDSS 2017-03-13T23:12:13     bias: 2017-03-14T09:41:34 (0.44) dome: r_SDSS 2017-03-14T09:41:34 (0.44)


In [12]:
#We associate a bpm partition to each dome ff frame
for dome_part in dome_partitions:
    dome_part.bpmPartition = bpm_partitions[0]
    for i in range(1,33):
        bpm_file = (dome_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, dome_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((dome_part.bpmPartition).base_path+'/toreduce_dome').touch()

    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(dome_date, bias_date)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part        
    
    mbias = (dome_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()

In [13]:
# here we associate biases to each dome partition
for dome_part in dome_partitions:
    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = timeDistance(dome_date, bias_date)
        
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part

    
    print('dome: %s %s'%(dome_part.filter, dome_part.OBstart,)+\
            '     bias: %s (%5.3f)'%((dome_part.biasPartition).OBstart, biasBestDeltaT/3600.0/24.,))
    
# at this point we have the biases  for each dome ff frame. Now we link
for dome_part in dome_partitions:
    mbias = (dome_part.biasPartition).base_path+'/OMEGACAM_MB_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/OMEGACAM_MB_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()
        

dome: r_SDSS 2017-03-14T09:41:34     bias: 2017-03-14T09:41:34 (0.000)


In [14]:
# Finalle associate a bpm partition to each bias frame
for bias_part in bias_partitions:
    bias_part.bpmPartition = bpm_partitions[0]
    for i in range(1,33):
        bpm_file = (bias_part.bpmPartition).base_path+'/OMEGACAM_BPM_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, bias_part.base_path+'/OMEGACAM_BPM_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((bias_part.bpmPartition).base_path+'/toreduce_bias').touch()
